# TabPFN-3 Regressor — DIMER task-inference tutorial (standalone)

[![GitHub](https://img.shields.io/badge/GitHub-181717?style=flat&logo=github&logoColor=white)](https://github.com/kurtvalcorza/tabpfn-regressor-pipeline) [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kurtvalcorza/tabpfn-regressor-pipeline/blob/main/tutorials/tabpfn_regressor_colab.ipynb) [![Hugging Face](https://img.shields.io/badge/%F0%9F%A4%97%20Hugging%20Face-Prior--Labs%2Ftabpfn__3-ffcc4d?style=flat)](https://huggingface.co/Prior-Labs/tabpfn_3) [![Upstream](https://img.shields.io/badge/Upstream-PriorLabs%2FTabPFN-181717?style=flat&logo=github&logoColor=white)](https://github.com/PriorLabs/TabPFN) [![arXiv](https://img.shields.io/badge/arXiv-2605.13986-b31b1b.svg)](https://arxiv.org/abs/2605.13986)

**Profile:** `TASK-INFERENCE`  
**Mode:** `GUIDED`  
**Notebook specification:** DIMER Notebook Specification 2.0 — **standalone** (§4)  
**Capability:** supervised tabular regression by **in-context learning** with the pinned `Prior-Labs/tabpfn_3` regressor checkpoint: the labelled training rows are the support set, no gradient update, point-estimate predictions with no shipped intervals, a `model.tabpfn_fit` + `model.ckpt` + `artifact_manifest.json` bundle reloaded across a fresh boundary

**This notebook is standalone.** It carries the repository's pipeline module (`src/tabpfn_regressor_pipeline/pipeline.py` at revision `b0f5ba58f9f8`) verbatim in Section 2, the pinned model identity and the per-file SHA-256 manifest in Section 3, and the exact runtime pins in Section 1, so it keeps working after export even if the repository changes or disappears. Its only external dependencies are the pinned Python distributions and the Hugging Face Hub at the immutable revision `24a16a89d245878b846555110985634aa2e656d7` (~233 MB, digest-verified before loading). It was generated by `tools/build_notebook.py` (build_notebook.py/2); edit the repository and regenerate rather than editing cells.

**Run all:** Selecting **Run all** in a fresh supported runtime installs the pinned dependencies, stages and digest-verifies the pinned TabPFN-3 checkpoint (an ungated download; the TabPFN-3 licence's non-commercial terms still apply to what you do with it), draws the deterministic synthetic regression table in code (600 rows, two numeric features and one categorical, seeded `train`/`val`/`test` split, no download), validates it into an input manifest, **fits the regressor in context on the training split** (support-row registration on the pinned checkpoint — TabPFN has no gradient route in this pipeline), evaluates on the held-out split against a training-mean baseline (MAE/RMSE) and writes the evaluation report, exports the artifact bundle and reloads it across a fresh boundary, scores new rows, and exports machine-readable results and provenance. No repository clone, DIMER worker or service, credential, upload dialog or configuration edit is required (NOTEBOOK_SPEC 2.0 §5).

**Bring Your Own Data:** Two optional branches, both off by default and never part of the default path: `USE_BYOD = True` (or `BYOD_ZIP_PATH`) in Section 4 supplies your own labelled table (a ZIP with `train.csv`/`val.csv`/`test.csv`, or a single `train.csv` that is split with a seeded hold-out) which enters the same validation, in-context fitting, evaluation, export and fresh-reload cells as the synthetic sample; `USE_BYOD_ROWS = True` (or `NEW_DATA_PATH`) in Section 9 predicts your own unlabelled rows (point predictions) with the reloaded estimator. Expected schema, ceilings and privacy guidance are stated in the Prerequisites and in those cells; uploads stay inside this runtime.

TabPFN-3 is a Transformer trained on a prior over synthetic tabular tasks so that it performs supervised regression in a single forward pass: `fit` registers your labelled training rows as the in-context support and performs **no gradient update**; query rows attend to that support and the head emits a predictive distribution whose mean is the point estimate. This notebook is **inference-only**: it runs that in-context path through the carried `tabpfn_regressor_pipeline` module against the pinned, digest-verified TabPFN-3 checkpoint. **The DIMER fine-tuning path of this pipeline is not carried here**: it lives in the private `tabpfn-regressor-finetuner` worker, and the TabPFN-3 weights are released under `tabpfn-3-license-v1.0`, whose Non-Commercial Purpose excludes production deployment — so this tutorial is testing-and-evaluation material and carries no private code. The carried module owns the pinned snapshot scheme, the input contract, the ICL fit / predict / evaluate calls, the artifact bundle the serving path consumes (with its pre-load validation) and the evaluation report. Sample metrics on synthetic data are tutorial sanity evidence only, not benchmark or production evidence; `predict` returns **point estimates only** — no prediction intervals or calibrated uncertainty are shipped.

**Learning objectives:** install the pinned runtime, read what the carried module guarantees, resolve and digest-verify the immutable TabPFN-3 checkpoint, generate the synthetic sample or supply a train/val/test ZIP through the archive-safety rules, validate it into an input manifest with a recorded rejection, fit in context and read MAE, RMSE, R² and MAPE against the training-mean baseline, export the artifact bundle and prove that it reloads across a fresh boundary and reproduces the recorded metric, predict genuinely new rows, and export machine-readable outputs plus provenance.

**This notebook does not demonstrate:** gradient fine-tuning (the private DIMER worker path, dropped from this standalone notebook), classification, prediction intervals or calibrated uncertainty, the v2 / v2.5 / v2.6 generations (only the pinned v3 checkpoint is carried), temporal or grouped splitting, or any quality claim beyond one holdout of one table.

## Prerequisites

- **Runtime:** Python 3.11+ (Google Colab or Jupyter). CPU is sufficient for the default sample; CUDA is used automatically when present. The pinned `torch==2.11.0` install is the largest download of the run; the checkpoint is 222 MiB.
- **Knowledge:** basic Python and pandas, the train/val/test convention, and what MAE, RMSE and R² mean.
- **Data:** the default path draws a deterministic synthetic table (600 rows, two numeric features and one categorical, `train.csv`/`val.csv`/`test.csv`) in code and needs no download and no private data; a gated BYOD path accepts one ZIP with the same layout (or a single `train.csv`, from which a seeded random holdout is drawn). Do not upload confidential or restricted data to a hosted notebook environment unless you are authorized to do so. Uploaded inputs remain in the notebook runtime; this pipeline does not send them to a third-party inference API.
- **Licence:** the TabPFN-3 weights are non-commercial (`tabpfn-3-license-v1.0`): testing, evaluation and internal benchmarking only. Clear the licence before any production use.
- **Credentials:** none. `Prior-Labs/tabpfn_3` is public and not access-gated.
- **External access:** the Hugging Face Hub only, to fetch the pinned `Prior-Labs/tabpfn_3` snapshot (~233 MB in total) at revision `24a16a89d245…`. No GitHub access and no credentials are required; nothing is installed from this repository.

## 1. Install the pinned runtime

The dependency set is pinned exactly (the same pins as the repository's pyproject.toml at the generating revision; any `--index-url`/`--find-links` lines are passed to pip as written) and installed directly — there is no repository clone and no package install. If a pin replaces a distribution this runtime has already imported, the cell stops with a restart instruction rather than continuing with mixed versions. Look for a dictionary reporting the notebook's source revision, Python, `torch`, `pandas`, `sklearn` versions, and whether CUDA is available.

In [ ]:
import importlib
import importlib.metadata
import os
import platform
import subprocess
import sys

PINS = [
    'torch==2.11.0',
    'tabpfn==8.1.0',
    'pandas==2.3.2',
    'scikit-learn==1.9.0',
    'huggingface-hub==0.36.2',
]
NOTEBOOK_SOURCE = {
    'repository': 'tabpfn-regressor-pipeline',
    'repository_revision': 'b0f5ba58f9f8e10a21ad276da553c6bd22f014c6',
    'embedded_module': 'src/tabpfn_regressor_pipeline/pipeline.py',
    'embedded_modules': ['src/tabpfn_regressor_pipeline/pipeline.py'],
    'module_sha256': 'e5499d350dc412a97abf431f50ad48fdea4c3dcad042baf487e47ee849f3bea7',
    'generator': 'build_notebook.py/2',
    'notebook_spec': '2.0',
}
SKIP_INSTALL = os.environ.get('DIMER_NOTEBOOK_CI_PREINSTALLED') == '1'

def _installed_version(distribution):
    try:
        return importlib.metadata.version(distribution)
    except importlib.metadata.PackageNotFoundError:
        return None

if not SKIP_INSTALL:
    # Capture every distribution already imported in this runtime, whatever its module name
    # (PIL -> pillow), so a pinned install that replaces a loaded package is detected and the
    # notebook stops with a restart instruction instead of continuing with mixed versions.
    _module_dists = importlib.metadata.packages_distributions()
    _loaded = sorted({d for m in list(sys.modules) for d in _module_dists.get(m.partition('.')[0], ())})
    loaded = {distribution: _installed_version(distribution) for distribution in _loaded}
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *PINS], check=True)
    importlib.invalidate_caches()
    stale = []
    for distribution, before in loaded.items():
        installed = _installed_version(distribution)
        if before is not None and before != installed:
            stale.append(f'{distribution}: loaded={before}, installed={installed}')
    if stale:
        raise RuntimeError('Core dependencies changed while older modules were loaded: ' + '; '.join(stale) + '. Restart the runtime, then rerun from the top.')

import torch, pandas, sklearn
print({'notebook_source': NOTEBOOK_SOURCE, 'python': platform.python_version(), 'torch': torch.__version__, 'pandas': pandas.__version__, 'sklearn': sklearn.__version__, 'cuda': torch.cuda.is_available()})

## 2. Pipeline code (carried verbatim from `src/tabpfn_regressor_pipeline/` @ `b0f5ba58f9f8`)

The next 1 cell(s) **are** the repository's package, module by module in dependency order: the pinned identity constants, snapshot verification (`verify_snapshot`), staged download (`stage_missing_files`), the named operational ceilings, the public validation and evaluation helpers, and the pipeline class. The text is the modules', byte for byte, except for the rewrite rules listed in `tools/build_notebook.py` (1 rule(s), plus the removal of package-relative `from .x import` lines, whose names are already defined by the preceding cells). The repository's parity test (`tests/test_notebook_parity.py`) fails whenever these cells and the modules diverge, so what you run here is what the repository tests. Nothing in these cells runs a model yet.

**Module 1/1:** `src/tabpfn_regressor_pipeline/pipeline.py`

In [ ]:
"""TabPFN-3 tabular regression — public tutorial API (DIMER pipeline, standalone-notebook carrier).

Inference-only: the pinned TabPFN-3 regressor checkpoint conditions on the labelled training rows **in context**
(``fit`` registers the support set; no gradient update) and predicts query rows in a forward pass. The DIMER
fine-tuning path of this pipeline lives in the private ``tabpfn-regressor-finetuner`` worker and is NOT carried
here (private code; TabPFN-3 weights are non-commercial). This module owns the pinned snapshot scheme, the input
contract (DAT24), the ICL fit / predict / evaluate calls, the ``model.tabpfn_fit`` + ``model.ckpt`` +
``artifact_manifest.json`` bundle the serving path consumes, its pre-load validation, and the evaluation report
(EVAL21). ``tabpfn`` / ``torch`` are imported lazily inside the functions that need them.
"""
# ruff: noqa: E501  -- contract dictionaries and messages are kept on single lines
from __future__ import annotations

import hashlib
import io
import json
import shutil
import tempfile
import zipfile
from collections.abc import Callable, Sequence
from dataclasses import dataclass
from pathlib import Path, PurePosixPath
from typing import Any

import numpy as np
import pandas as pd

MODEL_ID = "Prior-Labs/tabpfn_3"
MODEL_REVISION = "24a16a89d245878b846555110985634aa2e656d7"
# Non-commercial licence: testing, evaluation and internal benchmarking only (no production deployment).
MODEL_LICENSE = "tabpfn-3-license-v1.0"
MODEL_KEY = "tabpfn-3-regressor"
DEFAULT_WEIGHTS_DIR = Path.cwd() / "weights" / MODEL_KEY  # standalone rewrite (build_notebook.py): working-directory-relative
MANIFEST_NAME = "dimer-base-manifest.json"
WEIGHTS_FILE = "tabpfn-v3-regressor-v3_default.ckpt"
WEIGHTS_SHA256 = "311ce18d97e9533d8585eaadafe040fbdd8070533209ed8696641dadc97a7301"
WEIGHTS_BYTES = 233289807
MODEL_VERSION = "v3"
TABPFN_VERSION = "8.1.0"

TASK_TYPE = "tabular_regression"
# Ceilings of the selected generation (contract/model-versions.json, `v3`): rows, features (maxClasses is unused by regression).
MAX_TRAIN_ROWS = 1_000_000
MAX_FEATURES = 2000
MIN_TRAIN_ROWS = 10
DEFAULT_N_ESTIMATORS = 4
DEFAULT_SEED = 42
DEFAULT_VALIDATION_SPLIT = 0.2
MAX_ZIP_EXPANDED_BYTES = 512 * 1024 * 1024  # BYOD ZIP expansion ceiling (512 MiB)
MAX_ZIP_RATIO = 200  # compression-bomb guard: expanded / compressed
DECISION_RULE = "point-estimate"  # predict() returns TabPFN's point estimate (mean of the predictive distribution); no intervals are shipped
METRIC_IDS = ("mae", "rmse", "r2", "mape")
ARTIFACT_SCHEMA_VERSION = 1
FITTED_NAME = "model.tabpfn_fit"
CHECKPOINT_NAME = "model.ckpt"
ARTIFACT_MANIFEST_NAME = "artifact_manifest.json"


# --------------------------------------------------------------------------- snapshot scheme


def sha256_hex(data: bytes) -> str:
    return hashlib.sha256(data).hexdigest()


def sha256_file(path: str | Path) -> str:
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        for chunk in iter(lambda: handle.read(1 << 20), b""):
            digest.update(chunk)
    return digest.hexdigest()


def _read_manifest(root: Path) -> dict[str, Any]:
    manifest_path = root / MANIFEST_NAME
    if not manifest_path.is_file():
        raise FileNotFoundError(f"snapshot manifest missing: {manifest_path}")
    manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
    if (manifest.get("modelId"), manifest.get("revision")) != (MODEL_ID, MODEL_REVISION):
        raise ValueError(f"manifest names {manifest.get('modelId')}@{manifest.get('revision')}, expected {MODEL_ID}@{MODEL_REVISION}")
    if not isinstance(manifest.get("files"), list) or not manifest["files"]:
        raise ValueError("manifest has no file entries")
    return manifest


def verify_snapshot(weights_dir: str | Path | None = None) -> dict[str, Any]:
    """Re-hash every manifest entry under ``weights_dir``; raise on any missing file, size or digest mismatch."""
    root = Path(weights_dir or DEFAULT_WEIGHTS_DIR)
    manifest = _read_manifest(root)
    checkpoint_ok = False
    for entry in manifest["files"]:
        path = root / entry["path"]
        if not path.is_file():
            raise FileNotFoundError(f"snapshot file missing: {path}")
        size = path.stat().st_size
        if size != entry["bytes"]:
            raise ValueError(f"{entry['path']}: size {size} != manifest {entry['bytes']}")
        digest = sha256_file(path)
        if digest != entry["sha256"]:
            raise ValueError(f"{entry['path']}: sha256 {digest} != manifest {entry['sha256']}")
        if entry["path"] == WEIGHTS_FILE:
            checkpoint_ok = digest == WEIGHTS_SHA256 and size == WEIGHTS_BYTES
    if not checkpoint_ok:
        raise ValueError(f"manifest does not pin {WEIGHTS_FILE} at sha256 {WEIGHTS_SHA256} / {WEIGHTS_BYTES} bytes")
    return manifest


def stage_missing_files(weights_dir: str | Path | None = None, *, allow_download: bool = False, downloader: Callable[..., Any] | None = None) -> list[str]:
    """Fetch the manifest entries absent from ``weights_dir`` (revision-pinned, never ``main``); refuse without ``allow_download``."""
    root = Path(weights_dir or DEFAULT_WEIGHTS_DIR)
    manifest = _read_manifest(root)
    missing = [entry["path"] for entry in manifest["files"] if not (root / entry["path"]).is_file()]
    if not missing:
        return []
    if not allow_download:
        raise FileNotFoundError(f"snapshot at {root} is missing {missing}; pass allow_download=True to stage them from {MODEL_ID}@{MODEL_REVISION[:12]}")
    if downloader is None:
        from huggingface_hub import hf_hub_download

        downloader = hf_hub_download
    for rel in missing:
        downloader(repo_id=MODEL_ID, filename=rel, revision=MODEL_REVISION, local_dir=str(root))
        if not (root / rel).is_file():
            raise FileNotFoundError(f"download did not produce {root / rel}")
    return missing


# --------------------------------------------------------------------------- datasets


def build_synthetic_dataset(out: str | Path, rows: int = 600, seed: int = DEFAULT_SEED) -> Path:
    """Deterministic train/val/test regression ZIP (mirrors examples/build_synthetic_dataset.py); tutorial data only."""
    from sklearn.model_selection import train_test_split

    if rows < 100:
        raise ValueError("rows must be at least 100")
    rng = np.random.default_rng(seed)
    x1 = rng.normal(size=rows)
    x2 = rng.uniform(-2.0, 2.0, size=rows)
    category = rng.choice(["a", "b", "c"], size=rows)
    category_effect = pd.Series(category).map({"a": -2.0, "b": 0.5, "c": 3.0}).to_numpy()
    noise = rng.normal(scale=0.35, size=rows)
    target = 4.0 * x1 - 1.5 * x2 + category_effect + noise - 1.0
    frame = pd.DataFrame({"x1": x1, "x2": x2, "category": category, "target": target})
    train, remainder = train_test_split(frame, test_size=0.3, random_state=seed)
    val, test = train_test_split(remainder, test_size=0.5, random_state=seed)
    out = Path(out).resolve()
    out.parent.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(out, "w", compression=zipfile.ZIP_DEFLATED) as archive:
        for name, part in (("train.csv", train), ("val.csv", val), ("test.csv", test)):
            archive.writestr(name, part.reset_index(drop=True).to_csv(index=False))
    return out


def _safe_member(name: str) -> str:
    path = PurePosixPath(name.replace("\\", "/"))
    if path.is_absolute() or ".." in path.parts or not path.name:
        raise ValueError(f"unsafe archive member: {name!r}")
    return path.name


def safe_extract_zip(zip_path: str | Path, destination: str | Path, max_expanded_bytes: int = MAX_ZIP_EXPANDED_BYTES, max_ratio: int = MAX_ZIP_RATIO) -> list[str]:
    """Extract a ZIP member by member (bare file names only, no directories/absolute/traversing paths); never ``extractall``."""
    zip_path, destination = Path(zip_path), Path(destination)
    destination.mkdir(parents=True, exist_ok=True)
    written: list[str] = []
    with zipfile.ZipFile(zip_path) as archive:
        infos = [info for info in archive.infolist() if not info.is_dir()]
        expanded = sum(info.file_size for info in infos)
        compressed = max(1, sum(info.compress_size for info in infos))
        if expanded > max_expanded_bytes:
            raise ValueError(f"archive expands to {expanded} bytes > ceiling {max_expanded_bytes}")
        if expanded / compressed > max_ratio:
            raise ValueError(f"archive compression ratio {expanded / compressed:.0f} > ceiling {max_ratio}")
        for info in infos:
            name = _safe_member(info.filename)
            with archive.open(info) as source, open(destination / name, "wb") as target:
                shutil.copyfileobj(source, target)
            written.append(name)
    return written


def read_dataset_zip(zip_path: str | Path, *, max_expanded_bytes: int = MAX_ZIP_EXPANDED_BYTES, max_ratio: int = MAX_ZIP_RATIO) -> dict[str, pd.DataFrame]:
    """Read ``train.csv`` (+ optional ``val.csv`` / ``test.csv``) from a ZIP with archive-safety checks; never ``extractall``."""
    zip_path = Path(zip_path)
    frames: dict[str, pd.DataFrame] = {}
    with zipfile.ZipFile(zip_path) as archive:
        infos = [info for info in archive.infolist() if not info.is_dir()]
        expanded = sum(info.file_size for info in infos)
        compressed = max(1, sum(info.compress_size for info in infos))
        if expanded > max_expanded_bytes:
            raise ValueError(f"archive expands to {expanded} bytes > ceiling {max_expanded_bytes}")
        if expanded / compressed > max_ratio:
            raise ValueError(f"archive compression ratio {expanded / compressed:.0f} > ceiling {max_ratio}")
        for info in infos:
            name = _safe_member(info.filename)
            if name in ("train.csv", "val.csv", "test.csv"):
                if name in frames:
                    raise ValueError(f"archive carries {name} more than once")
                frames[name] = pd.read_csv(io.BytesIO(archive.read(info.filename)))
    if "train.csv" not in frames:
        raise ValueError("archive has no train.csv")
    return frames


def read_dataset_dir(dataset_dir: str | Path) -> dict[str, pd.DataFrame]:
    """Read a ZIP (``*.zip``) or loose ``train.csv`` / ``val.csv`` / ``test.csv`` from a directory."""
    root = Path(dataset_dir)
    zips = sorted(root.glob("*.zip"))
    if zips:
        if len(zips) > 1:
            raise ValueError(f"expected one ZIP in {root}, found {len(zips)}")
        return read_dataset_zip(zips[0])
    frames = {name: pd.read_csv(root / name) for name in ("train.csv", "val.csv", "test.csv") if (root / name).is_file()}
    if "train.csv" not in frames:
        raise ValueError(f"{root} has neither a ZIP nor train.csv")
    return frames


def random_holdout(train: pd.DataFrame, target_column: str, validation_split: float = DEFAULT_VALIDATION_SPLIT, seed: int = DEFAULT_SEED) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Seeded random split for a single ``train.csv``; wrong for temporal/grouped data (supply explicit splits instead)."""
    from sklearn.model_selection import train_test_split

    if not 0.0 < validation_split < 1.0:
        raise ValueError("validation_split must be in (0, 1)")
    if target_column not in train.columns:
        raise ValueError(f"target column {target_column!r} not present")
    fit_part, val_part = train_test_split(train, test_size=validation_split, random_state=seed)
    return fit_part.reset_index(drop=True), val_part.reset_index(drop=True)


# --------------------------------------------------------------------------- input contract (DAT24)

INPUT_SCHEMA = {
    "format": "CSV table(s): train.csv with an optional val.csv / test.csv (ZIP or loose files); one row per observation",
    "target": "one numeric target column (declared name), no missing or non-finite values, >= MIN_TRAIN_ROWS training rows, non-constant",
    "features": "every other column; numeric or categorical (strings); unique column names; identical schema across splits",
    "ceilings": {"MAX_TRAIN_ROWS": MAX_TRAIN_ROWS, "MAX_FEATURES": MAX_FEATURES, "MIN_TRAIN_ROWS": MIN_TRAIN_ROWS},
    "splits": "explicit val.csv/test.csv are preserved; without val.csv a seeded random holdout is drawn (independent rows assumed)",
    "reserved_columns": "no `prediction` column in inputs",
}


class InputRejected(ValueError):
    """Raised by ``validate_inputs`` with a structured finding."""

    def __init__(self, finding: dict[str, Any]) -> None:
        super().__init__(finding["message"])
        self.finding = finding


def _reject(code: str, message: str, observed: Any = None) -> InputRejected:
    return InputRejected({"code": code, "verdict": "rejected", "message": message, "observed": observed})


def _check_frame(name: str, frame: pd.DataFrame, target_column: str, feature_columns: list[str] | None) -> list[str]:
    columns = list(frame.columns)
    duplicates = sorted({c for c in columns if columns.count(c) > 1})
    if duplicates:
        raise _reject("DUPLICATE_COLUMNS", f"{name}: duplicate column names", duplicates)
    if target_column not in columns:
        raise _reject("TARGET_MISSING", f"{name}: target column {target_column!r} not present", columns)
    features = [c for c in columns if c != target_column]
    reserved = [c for c in features if c == "prediction"]
    if reserved:
        raise _reject("RESERVED_COLUMNS", f"{name}: reserved output columns present", reserved)
    if feature_columns is not None and features != feature_columns:
        raise _reject("SCHEMA_MISMATCH", f"{name}: feature columns differ from train.csv", {"expected": feature_columns, "observed": features})
    if frame[target_column].isna().any():
        raise _reject("TARGET_MISSING_VALUES", f"{name}: target has missing values", int(frame[target_column].isna().sum()))
    if not pd.api.types.is_numeric_dtype(frame[target_column]):
        raise _reject("TARGET_NOT_NUMERIC", f"{name}: target column {target_column!r} is not numeric", str(frame[target_column].dtype))
    if not np.isfinite(frame[target_column].to_numpy(dtype=float)).all():
        raise _reject("TARGET_NON_FINITE", f"{name}: target has infinite values", target_column)
    if len(frame) == 0:
        raise _reject("EMPTY_SPLIT", f"{name}: no rows", 0)
    return features


def validate_inputs(train: pd.DataFrame, target_column: str, *, val: pd.DataFrame | None = None, test: pd.DataFrame | None = None, names: Sequence[str] | None = None) -> dict[str, Any]:
    """Apply the input contract to the supplied splits and return the DAT24 input manifest (raises ``InputRejected``)."""
    features = _check_frame("train.csv", train, target_column, None)
    if not features:
        raise _reject("NO_FEATURES", "train.csv has no feature columns", list(train.columns))
    if len(features) > MAX_FEATURES:
        raise _reject("TOO_MANY_FEATURES", f"train.csv has {len(features)} features > MAX_FEATURES {MAX_FEATURES}", len(features))
    if len(train) > MAX_TRAIN_ROWS:
        raise _reject("TOO_MANY_ROWS", f"train.csv has {len(train)} rows > MAX_TRAIN_ROWS {MAX_TRAIN_ROWS}", len(train))
    if len(train) < MIN_TRAIN_ROWS:
        raise _reject("TOO_FEW_ROWS", f"train.csv has {len(train)} rows < MIN_TRAIN_ROWS {MIN_TRAIN_ROWS}", len(train))
    target_values = train[target_column].to_numpy(dtype=float)
    if float(np.std(target_values)) == 0.0:
        raise _reject("CONSTANT_TARGET", "train.csv target is constant; nothing to regress", float(target_values[0]))
    findings: list[dict[str, Any]] = []
    target_stats = {"mean": float(np.mean(target_values)), "std": float(np.std(target_values)), "min": float(np.min(target_values)), "max": float(np.max(target_values))}
    splits: dict[str, Any] = {"train": {"rows": int(len(train)), "target": target_stats}}
    for name, frame in (("val.csv", val), ("test.csv", test)):
        if frame is None:
            continue
        _check_frame(name, frame, target_column, features)
        values = frame[target_column].to_numpy(dtype=float)
        if values.min() < target_stats["min"] or values.max() > target_stats["max"]:
            findings.append({"input": name, "verdict": "warning", "code": "TARGET_OUT_OF_TRAINING_RANGE", "message": f"{name} targets fall outside the training range; extrapolation is being scored", "observed": {"min": float(values.min()), "max": float(values.max())}})
        splits[name.split(".")[0]] = {"rows": int(len(frame)), "target": {"mean": float(np.mean(values)), "std": float(np.std(values)), "min": float(values.min()), "max": float(values.max())}}
    numeric = [c for c in features if pd.api.types.is_numeric_dtype(train[c])]
    for c in numeric:
        values = train[c].dropna().to_numpy(dtype=float)
        if values.size and not np.isfinite(values).all():
            raise _reject("NON_FINITE_FEATURE", f"train.csv numeric feature {c!r} contains infinite values", c)
    if (target_values == 0).mean() > 0.5:
        findings.append({"input": "train.csv", "verdict": "warning", "code": "ZERO_HEAVY_TARGET", "message": "more than half of the training targets are zero; mape is computed over non-zero rows only", "observed": float((target_values == 0).mean())})
    return {
        "schema": dict(INPUT_SCHEMA),
        "inputs": [{"id": names[0] if names else "dataset-0", "mode": "in-context-fit", "target_column": target_column, "feature_columns": features, "numeric_features": len(numeric), "categorical_features": len(features) - len(numeric), "target_stats": target_stats, "splits": splits, "train_sha256": sha256_hex(train.to_csv(index=False).encode("utf-8"))}],
        "verdict": "accepted",
        "findings": findings,
        "model_id": MODEL_ID,
        "model_revision": MODEL_REVISION,
        "model_version": MODEL_VERSION,
    }


def validate_new_rows(frame: pd.DataFrame, feature_columns: Sequence[str], *, target_column: str | None = None) -> pd.DataFrame:
    """Rows for inference must carry exactly the artifact's feature columns and no target/output columns."""
    columns = list(frame.columns)
    duplicates = sorted({c for c in columns if columns.count(c) > 1})
    if duplicates:
        raise _reject("DUPLICATE_COLUMNS", "new rows: duplicate column names", duplicates)
    reserved = [c for c in [target_column, "prediction"] if c and c in columns]
    if reserved:
        raise _reject("RESERVED_COLUMNS", "new rows: remove target/prediction columns before inference", reserved)
    missing = [c for c in feature_columns if c not in columns]
    extra = [c for c in columns if c not in feature_columns]
    if missing or extra:
        raise _reject("SCHEMA_MISMATCH", "new rows: feature schema mismatch", {"missing": missing, "extra": extra})
    if len(frame) == 0:
        raise _reject("EMPTY_INPUT", "new rows: no rows", 0)
    frame = frame.loc[:, list(feature_columns)].copy()
    for c in frame.select_dtypes(include=np.number).columns:
        values = frame[c].dropna().to_numpy(dtype=float)
        if values.size and not np.isfinite(values).all():
            raise _reject("NON_FINITE_FEATURE", f"new rows: numeric feature {c!r} contains infinite values", c)
    return frame


# --------------------------------------------------------------------------- metrics / baseline


def regression_metrics(y_true: Sequence[Any], y_pred: Sequence[Any]) -> dict[str, float]:
    """MAE, RMSE, R² and MAPE (the latter over non-zero targets only; absent when every target is zero)."""
    from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

    yt = np.asarray(y_true, dtype=float)
    yp = np.asarray(y_pred, dtype=float)
    if yt.shape != yp.shape or yt.size == 0:
        raise ValueError("y_true and y_pred must be equal-length, non-empty 1-D arrays")
    out = {"mae": float(mean_absolute_error(yt, yp)), "rmse": float(np.sqrt(mean_squared_error(yt, yp))), "r2": float(r2_score(yt, yp)) if yt.size > 1 and float(np.std(yt)) > 0 else float("nan")}
    nonzero = yt != 0
    if nonzero.any():
        out["mape"] = float(np.mean(np.abs((yt[nonzero] - yp[nonzero]) / yt[nonzero])))
    return out


def mean_baseline(train_targets: Sequence[Any], eval_targets: Sequence[Any]) -> dict[str, Any]:
    """Always predict the training mean, scored on the evaluation rows."""
    train_targets = np.asarray(train_targets, dtype=float)
    eval_targets = np.asarray(eval_targets, dtype=float)
    if train_targets.size == 0 or eval_targets.size == 0:
        raise ValueError("baseline needs training and evaluation targets")
    mean = float(train_targets.mean())
    metrics = regression_metrics(eval_targets, np.full(eval_targets.shape, mean))
    return {"trainMean": mean, **metrics}


# --------------------------------------------------------------------------- artifact bundle


def _safe_fitted_archive(path: Path) -> dict[str, Any]:
    with zipfile.ZipFile(path) as archive:
        names = archive.namelist()
        for name in names:
            _safe_member(name)
        if "init_params.json" not in names:
            raise ValueError("fitted archive lacks init_params.json")
        return json.loads(archive.read("init_params.json"))


def manifest_digest(manifest: dict[str, Any], key: str) -> str:
    """Both manifest shapes: flat ``<key>Sha256`` (this module, the worker) or nested ``sha256[<key>]``."""
    value = manifest.get(key + "Sha256") or (manifest.get("sha256") or {}).get(key)
    if not isinstance(value, str) or len(value) != 64:
        raise ValueError(f"manifest has no SHA-256 for {key}")
    return value


def validate_artifact_bundle(artifact_dir: str | Path, *, expected_fitted_sha256: str = "", expected_checkpoint_sha256: str = "") -> dict[str, Any]:
    """Check manifest schema, member names, sizes, digests and archive safety BEFORE any model state is deserialised."""
    root = Path(artifact_dir)
    manifest_path = root / ARTIFACT_MANIFEST_NAME
    if not manifest_path.is_file():
        raise FileNotFoundError(f"{ARTIFACT_MANIFEST_NAME} missing in {root}")
    manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
    if manifest.get("schemaVersion") != ARTIFACT_SCHEMA_VERSION or manifest.get("taskType") != TASK_TYPE:
        raise ValueError(f"unsupported manifest: schemaVersion={manifest.get('schemaVersion')} taskType={manifest.get('taskType')}")
    for key in ("targetColumn", "featureColumns"):
        if key not in manifest:
            raise ValueError(f"manifest lacks {key}")
    digests: dict[str, str] = {}
    for key, override in (("fittedEstimator", expected_fitted_sha256), ("foundationCheckpoint", expected_checkpoint_sha256)):
        name = manifest.get(key)
        if not isinstance(name, str) or Path(name).name != name:
            raise ValueError(f"manifest {key} must be a bare file name, got {name!r}")
        path = root / name
        if not path.is_file() or path.stat().st_size < 1024:
            raise ValueError(f"{name} is missing or implausibly small")
        digest = sha256_file(path)
        if digest != manifest_digest(manifest, key):
            raise ValueError(f"{name} SHA-256 {digest} != manifest {manifest_digest(manifest, key)}")
        if override and digest != override.strip().lower():
            raise ValueError(f"{name} SHA-256 {digest} != expected {override.strip().lower()}")
        digests[key] = digest
    init_params = _safe_fitted_archive(root / manifest["fittedEstimator"])
    return {**manifest, "verifiedSha256": digests, "recordedModelPath": init_params.get("model_path")}


def rewrite_model_path(fitted_archive: Path, checkpoint: Path, destination: Path) -> Path:
    """Copy a ``.tabpfn_fit`` archive while pointing its ``init_params.json`` ``model_path`` at ``checkpoint`` (original untouched)."""
    fitted_archive, checkpoint, destination = fitted_archive.resolve(), checkpoint.resolve(), destination.resolve()
    if not fitted_archive.is_file():
        raise FileNotFoundError(f"fitted estimator not found: {fitted_archive}")
    if not checkpoint.is_file():
        raise FileNotFoundError(f"companion checkpoint not found: {checkpoint}")
    saw_init = False
    destination.parent.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(fitted_archive, "r") as source, zipfile.ZipFile(destination, "w", compression=zipfile.ZIP_DEFLATED) as target:
        for info in source.infolist():
            _safe_member(info.filename)
            payload = source.read(info.filename)
            if info.filename == "init_params.json":
                params = json.loads(payload.decode("utf-8"))
                params["model_path"] = str(checkpoint)
                payload = json.dumps(params, sort_keys=True).encode("utf-8")
                saw_init = True
            target.writestr(info, payload)
    if not saw_init:
        destination.unlink(missing_ok=True)
        raise ValueError("fitted artifact does not contain init_params.json")
    return destination


def zip_artifact_bundle(artifact_dir: str | Path, zip_path: str | Path) -> str:
    """Zip the three bundle members flat (bare names) for transport to the artifact-inference notebook; returns the ZIP SHA-256."""
    root, zip_path = Path(artifact_dir), Path(zip_path)
    zip_path.parent.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as archive:
        for name in (ARTIFACT_MANIFEST_NAME, FITTED_NAME, CHECKPOINT_NAME):
            archive.write(root / name, arcname=name)
    return sha256_file(zip_path)


def _coerce_non_json_init_params(model: Any) -> None:
    """tabpfn 8.1.0 serialises ``get_params()`` as JSON; stringify the values that are not JSON-encodable (e.g. a Path)."""
    for param, value in model.get_params(deep=False).items():
        try:
            json.dumps(value)
        except (TypeError, ValueError):
            setattr(model, param, str(value))


# --------------------------------------------------------------------------- pipeline


@dataclass
class TabPFNRegressorPipeline:
    """The verified TabPFN-3 checkpoint plus, after ``fit`` or ``from_artifact``, an in-context-fitted estimator."""

    weights_path: Path
    device: str = "cpu"
    source: str = "local-snapshot"
    n_estimators: int = DEFAULT_N_ESTIMATORS
    random_state: int = DEFAULT_SEED
    target_column: str | None = None
    feature_columns: list[str] | None = None
    target_stats: dict[str, float] | None = None
    _model: Any = None

    @classmethod
    def from_pretrained(cls, device: str | None = None, weights_dir: str | Path | None = None, allow_download: bool = False, *, n_estimators: int = DEFAULT_N_ESTIMATORS, random_state: int = DEFAULT_SEED) -> TabPFNRegressorPipeline:
        root = Path(weights_dir or DEFAULT_WEIGHTS_DIR)
        stage_missing_files(root, allow_download=allow_download)
        verify_snapshot(root)
        if device is None:
            import torch

            device = "cuda" if torch.cuda.is_available() else "cpu"
        return cls(weights_path=root / WEIGHTS_FILE, device=device, n_estimators=n_estimators, random_state=random_state)

    def _build_estimator(self) -> Any:
        from tabpfn import TabPFNRegressor

        return TabPFNRegressor(model_path=str(self.weights_path), device=self.device, n_estimators=self.n_estimators, random_state=self.random_state, show_progress_bar=False)

    def fit(self, X: pd.DataFrame, y: Sequence[Any], *, target_column: str = "target", estimator_factory: Callable[[], Any] | None = None) -> TabPFNRegressorPipeline:
        """In-context fit: register the training rows as the support set. No gradient update is performed."""
        model = (estimator_factory or self._build_estimator)()
        targets = pd.Series(np.asarray(list(y), dtype=float), name=target_column)
        model.fit(X, targets)
        self._model = model
        self.target_column = target_column
        self.feature_columns = list(X.columns)
        values = targets.to_numpy()
        self.target_stats = {"mean": float(values.mean()), "std": float(values.std()), "min": float(values.min()), "max": float(values.max())}
        self.source = "in-context-fit"
        return self

    def _require(self) -> Any:
        if self._model is None or self.feature_columns is None:
            raise RuntimeError("no fitted estimator: call fit() or from_artifact() first")
        return self._model

    def predict_values(self, X: pd.DataFrame) -> np.ndarray:
        model = self._require()
        X = validate_new_rows(X, self.feature_columns or [], target_column=self.target_column)
        return np.asarray(model.predict(X), dtype=float).reshape(-1)

    def predict(self, X: pd.DataFrame) -> pd.DataFrame:
        """``prediction`` = TabPFN's point estimate per row (no intervals are shipped)."""
        values = self.predict_values(X)
        out = pd.DataFrame({"row_id": np.asarray(X.index), "prediction": values})
        out.attrs["decision_rule"] = DECISION_RULE
        return out

    def evaluate(self, X: pd.DataFrame, y: Sequence[Any]) -> dict[str, float]:
        return regression_metrics(np.asarray(list(y), dtype=float), self.predict_values(X))

    def save_artifact(self, artifact_dir: str | Path, *, saver: Callable[[Any, Path], None] | None = None) -> dict[str, Any]:
        """Write ``model.tabpfn_fit`` + ``model.ckpt`` (byte copy of the verified checkpoint) + ``artifact_manifest.json``."""
        model = self._require()
        root = Path(artifact_dir)
        root.mkdir(parents=True, exist_ok=True)
        if saver is None:
            from tabpfn.model_loading import save_fitted_tabpfn_model

            def saver(estimator: Any, path: Path) -> None:
                _coerce_non_json_init_params(estimator)
                save_fitted_tabpfn_model(estimator, path)

        saver(model, root / FITTED_NAME)
        shutil.copyfile(self.weights_path, root / CHECKPOINT_NAME)
        manifest = {
            "schemaVersion": ARTIFACT_SCHEMA_VERSION,
            "taskType": TASK_TYPE,
            "targetColumn": self.target_column,
            "featureColumns": list(self.feature_columns or []),
            "targetStats": dict(self.target_stats or {}),
            "fittedEstimator": FITTED_NAME,
            "foundationCheckpoint": CHECKPOINT_NAME,
            "fittedEstimatorSha256": sha256_file(root / FITTED_NAME),
            "foundationCheckpointSha256": sha256_file(root / CHECKPOINT_NAME),
            "portableLoader": "tabpfn_regressor_pipeline.TabPFNRegressorPipeline.from_artifact",
            "baseModel": {"modelId": MODEL_ID, "revision": MODEL_REVISION, "file": WEIGHTS_FILE, "sha256": WEIGHTS_SHA256, "modelVersion": MODEL_VERSION, "license": MODEL_LICENSE},
            "mode": "zero-shot-icl",
            "nEstimators": self.n_estimators,
            "randomState": self.random_state,
        }
        (root / ARTIFACT_MANIFEST_NAME).write_text(json.dumps(manifest, indent=2) + "\n", encoding="utf-8")
        return manifest

    @classmethod
    def from_artifact(cls, artifact_dir: str | Path, device: str | None = None, *, expected_fitted_sha256: str = "", expected_checkpoint_sha256: str = "", loader: Callable[[Path, str], Any] | None = None) -> TabPFNRegressorPipeline:
        """Validate the bundle, then reconstruct the estimator from the fitted archive + companion checkpoint (no refit, no download)."""
        root = Path(artifact_dir)
        manifest = validate_artifact_bundle(root, expected_fitted_sha256=expected_fitted_sha256, expected_checkpoint_sha256=expected_checkpoint_sha256)
        if device is None:
            import torch

            device = "cuda" if torch.cuda.is_available() else "cpu"
        if loader is None:
            from tabpfn.model_loading import load_fitted_tabpfn_model

            def loader(fitted: Path, dev: str) -> Any:
                return load_fitted_tabpfn_model(fitted, device=dev)

        checkpoint = root / manifest["foundationCheckpoint"]
        with tempfile.TemporaryDirectory() as temp_dir:
            rewritten = rewrite_model_path(root / manifest["fittedEstimator"], checkpoint, Path(temp_dir) / FITTED_NAME)
            model = loader(rewritten, device)
        if not hasattr(model, "predict"):
            raise RuntimeError("reconstructed estimator has no predict method")
        return cls(weights_path=checkpoint, device=device, source="artifact", n_estimators=int(manifest.get("nEstimators", DEFAULT_N_ESTIMATORS)), random_state=int(manifest.get("randomState", DEFAULT_SEED)), target_column=manifest["targetColumn"], feature_columns=list(manifest["featureColumns"]), target_stats=dict(manifest.get("targetStats") or {}), _model=model)


# --------------------------------------------------------------------------- evaluation report (EVAL21)


def evaluation_report(metrics: dict[str, float] | None, *, baseline: dict[str, Any] | None = None, n_validation: int = 0, target_column: str | None = None, sample_kind: str = "synthetic", reload_check: dict[str, Any] | None = None, split_name: str = "val.csv") -> dict[str, Any]:
    """Evaluation stage: the metrics are honest about what they are (single holdout, tutorial sample) or ``not-measurable``."""
    base = {
        "task": TASK_TYPE,
        "model_id": MODEL_ID,
        "model_revision": MODEL_REVISION,
        "model_version": MODEL_VERSION,
        "decision_rule": DECISION_RULE,
        "adaptation": "in-context conditioning only (no gradient update); the private-worker fine-tune path is not carried",
        "sample_kind": sample_kind,
        "n_validation": int(n_validation),
        "split": split_name,
        "target_column": target_column,
        "reload_check": reload_check,
        "caveats": ["single holdout, no dispersion estimate", "point estimates only; no prediction intervals are shipped", "mape is computed over non-zero targets only", "synthetic sample metrics are plumbing evidence only" if sample_kind == "synthetic" else "BYOD metrics are one holdout of one table"],
    }
    if not metrics or n_validation == 0:
        return {**base, "metrics": [], "verdict": "not-measurable", "reason": "no labelled validation split was scored", "needs": "a labelled, leakage-safe holdout from the deployment domain scored with mae / rmse / r2 against mean_baseline; repeated splits for any dispersion estimate"}
    entries = [{"id": k, "value": float(v)} for k, v in metrics.items() if k in METRIC_IDS]
    return {**base, "metrics": entries, "baselines": {"training_mean": baseline} if baseline else {}, "verdict": "sample-sanity", "reason": f"{n_validation} validation row(s) from one holdout; tutorial evidence, not a benchmark", "needs": "a domain-representative labelled test set, subgroup breakdowns, residual analysis on held-out data and repeated splits for any generalisable claim"}

## 3. Pin, stage and verify the model

The model identity is carried twice — `MODEL_ID`/`MODEL_REVISION` in the module above and the `4`-file manifest below (paths, byte sizes, SHA-256) — and the cell first asserts they agree. It writes the manifest into the working-directory snapshot, then `stage_missing_files(..., allow_download=True)` fetches exactly the entries that are absent from the Hugging Face Hub **at revision `24a16a89d245…`** (never `main`), `verify_snapshot` re-hashes every file and raises on the first size or digest mismatch, and only then does `TabPFNRegressorPipeline.from_pretrained(weights_dir=WEIGHTS_DIR)` load the verified files. There is no fallback to a different download and no remote model code is executed. The effective identity, device and weight source are printed before any inference.

In [ ]:
import json

MANIFEST = {
  "format": "dimer_hf_snapshot",
  "formatVersion": 1,
  "modelKey": "tabpfn-3-regressor",
  "modelId": "Prior-Labs/tabpfn_3",
  "revision": "24a16a89d245878b846555110985634aa2e656d7",
  "files": [
    {
      "path": "LICENSE",
      "bytes": 16794,
      "sha256": "dca491280b68f471312a15d54add7b8e724adf19fb0e113544b1ef91e060f5d5"
    },
    {
      "path": "README.md",
      "bytes": 8377,
      "sha256": "53d61ea9bbf605c42892d13c53afb78e5ba8a6d1513d89db86736a620d3c9e81"
    },
    {
      "path": "config.json",
      "bytes": 33,
      "sha256": "d9bc48f72a18bcbbb0a58dbe1ca7ac4123b9cfa1b7c0a79da5bcf3543cb32344"
    },
    {
      "path": "tabpfn-v3-regressor-v3_default.ckpt",
      "bytes": 233289807,
      "sha256": "311ce18d97e9533d8585eaadafe040fbdd8070533209ed8696641dadc97a7301"
    }
  ],
  "totalBytes": 233315011
}

if (MANIFEST['modelId'], MANIFEST['revision']) != (MODEL_ID, MODEL_REVISION):
    raise RuntimeError('inline manifest does not name the identity carried by the pipeline module; the notebook was not regenerated after a change')
WEIGHTS_DIR = DEFAULT_WEIGHTS_DIR
WEIGHTS_DIR.mkdir(parents=True, exist_ok=True)
with open(WEIGHTS_DIR / MANIFEST_NAME, 'w', encoding='utf-8') as handle:
    json.dump(MANIFEST, handle, indent=2)
print({'model_id': MODEL_ID, 'revision': MODEL_REVISION, 'license': MODEL_LICENSE, 'files': len(MANIFEST['files']), 'total_bytes': MANIFEST['totalBytes']})
fetched = stage_missing_files(WEIGHTS_DIR, allow_download=True)
print({'weights_dir': str(WEIGHTS_DIR), 'fetched': fetched})
snapshot = verify_snapshot(WEIGHTS_DIR)
_files = snapshot.get('files', []) if isinstance(snapshot, dict) else []
print({'verified_files': len(_files) if isinstance(_files, list) else _files, 'revision': snapshot.get('revision', MODEL_REVISION) if isinstance(snapshot, dict) else MODEL_REVISION})
pipe = TabPFNRegressorPipeline.from_pretrained(weights_dir=WEIGHTS_DIR)
print({'device': getattr(pipe, 'device', None), 'source': getattr(pipe, 'source', 'local-snapshot')})

## 4. Prepare the dataset: default synthetic sample or bring your own

The expected input is one `train.csv` with a declared numeric target column, plus optional `val.csv` and `test.csv` with identical columns (explicit splits are preserved, never re-split). Column names must be unique, the target must be numeric, finite, complete and non-constant with at least `MIN_TRAIN_ROWS` rows, and numeric features must be finite; every other column is a feature (numeric or string categorical). With `USE_BYOD = False` the carried module draws the deterministic synthetic sample (`build_synthetic_dataset`: a linear signal with a categorical effect and Gaussian noise, seed `SAMPLE_SEED`) as a ZIP; with `USE_BYOD = True` a ZIP is taken from `BYOD_ZIP_PATH` (an executor places it there) or uploaded, and read member by member by `read_dataset_zip` with the archive-safety rules (bare file names only, expanded-size and compression-ratio ceilings; never `extractall`). Without a `val.csv`, `random_holdout` draws a seeded holdout of `VALIDATION_SPLIT` — correct only for independent rows; temporal, grouped or patient-level data need your own leakage-safe splits. Look for the split sizes and the target summary.

In [ ]:
USE_BYOD = False  # @param {type:"boolean"}
BYOD_ZIP_PATH = ''  # @param {type:"string"}
TARGET_COLUMN = 'target'  # @param {type:"string"}
VALIDATION_SPLIT = 0.2  # @param {type:"number"}
SAMPLE_SEED = 42  # @param {type:"integer"}

WORK = Path('work')
shutil.rmtree(WORK, ignore_errors=True)
DATASET_DIR = WORK / 'dataset'
DATASET_DIR.mkdir(parents=True)
if USE_BYOD:
    if BYOD_ZIP_PATH:
        zip_name, zip_payload = Path(BYOD_ZIP_PATH).name, Path(BYOD_ZIP_PATH).read_bytes()
    else:
        from google.colab import files
        uploaded = files.upload()
        if len(uploaded) != 1:
            raise RuntimeError('Upload exactly one dataset ZIP.')
        zip_name, zip_payload = next(iter(uploaded.items()))
    zip_path = DATASET_DIR / Path(zip_name).name
    zip_path.write_bytes(zip_payload)
    dataset_origin = {'type': 'user-supplied ZIP (BYOD)', 'zip': zip_path.name}
    sample_kind = 'BYOD'
else:
    zip_path = build_synthetic_dataset(DATASET_DIR / 'synthetic.zip', rows=600, seed=SAMPLE_SEED)
    dataset_origin = {'type': 'deterministic synthetic tutorial sample', 'generator': 'tabpfn_regressor_pipeline.build_synthetic_dataset', 'rows': 600, 'seed': SAMPLE_SEED}
    sample_kind = 'synthetic'
dataset_sha256 = sha256_file(zip_path)
frames = read_dataset_zip(zip_path)
train_df = frames['train.csv']
val_df, test_df = frames.get('val.csv'), frames.get('test.csv')
split_origin = 'explicit train/val/test from the ZIP'
if val_df is None:
    train_df, val_df = random_holdout(train_df, TARGET_COLUMN, VALIDATION_SPLIT, seed=SAMPLE_SEED)
    split_origin = f'seeded random holdout of {VALIDATION_SPLIT} drawn from train.csv (independent rows assumed)'
print({'sample_kind': sample_kind, **dataset_origin, 'zip_sha256': dataset_sha256[:16], 'splits': split_origin})
print({'train': train_df.shape, 'val': val_df.shape, 'test': None if test_df is None else test_df.shape})
print('target summary (train):', train_df[TARGET_COLUMN].describe().round(3).to_dict())

## 5. Validate the inputs → input manifest

`validate_inputs` is the module's public validation stage: it applies the input contract (unique columns, declared target present, numeric, finite and non-constant, `MIN_TRAIN_ROWS` and the ceilings of the v3 generation — `MAX_TRAIN_ROWS`, `MAX_FEATURES` — identical schema across splits, no reserved `prediction` column, finite numeric features) and returns an **input manifest** naming the schema, the feature columns, the target statistics, the per-split row counts and target ranges, a digest of the training table and the verdict; warnings (validation targets outside the training range, a zero-heavy target that makes MAPE fragile) are carried as non-fatal findings. It is written to `outputs/tabpfn_regressor_input_manifest.json`. To show what rejection looks like, the cell also validates a probe table whose target column was renamed and records the structured finding. The ceilings and the decision rule are printed before any model runs.

In [ ]:
os.makedirs('outputs', exist_ok=True)
print({'ceilings': {'MAX_TRAIN_ROWS': MAX_TRAIN_ROWS, 'MAX_FEATURES': MAX_FEATURES, 'MIN_TRAIN_ROWS': MIN_TRAIN_ROWS}, 'decision_rule': DECISION_RULE, 'model_version': MODEL_VERSION})
input_manifest = validate_inputs(train_df, TARGET_COLUMN, val=val_df, test=test_df, names=[dataset_origin['type']])
try:
    validate_inputs(train_df.rename(columns={TARGET_COLUMN: 'label'}), TARGET_COLUMN)
except InputRejected as exc:
    input_manifest['findings'].append({'input': 'renamed-target-probe', **exc.finding})
with open('outputs/tabpfn_regressor_input_manifest.json', 'w', encoding='utf-8') as handle:
    json.dump(input_manifest, handle, indent=2, ensure_ascii=False, default=str)
FEATURE_COLUMNS = input_manifest['inputs'][0]['feature_columns']
print(json.dumps({k: input_manifest['inputs'][0][k] for k in ('id', 'mode', 'target_column', 'target_stats', 'numeric_features', 'categorical_features', 'splits')}, indent=2))
print('findings:', json.dumps(input_manifest['findings'], indent=2, default=str))

## 6. Fit in context and evaluate

`pipe.fit` builds `tabpfn.TabPFNRegressor` on the digest-verified checkpoint of Section 3 (`model_path` = the staged file, `N_ESTIMATORS` ensemble members, seed `SEED`) and registers the training rows as the in-context support — **no gradient update happens**; a run takes seconds on CPU for the sample. `pipe.evaluate` scores the frozen validation split (and `test.csv` when present): `mae` is the mean absolute error in target units; `rmse` weights large errors more; `r2` is the fraction of target variance explained (0 is the training-mean baseline, negative is worse than it); `mape` is the mean absolute percentage error over non-zero targets only. These are single-split numbers with no dispersion estimate. Look for the metrics and the reported device.

In [ ]:
N_ESTIMATORS = 4  # @param {type:"integer"}
SEED = 42  # @param {type:"integer"}

pipe.n_estimators, pipe.random_state = N_ESTIMATORS, SEED
pipe.fit(train_df[FEATURE_COLUMNS], train_df[TARGET_COLUMN], target_column=TARGET_COLUMN)
validation_metrics = pipe.evaluate(val_df[FEATURE_COLUMNS], val_df[TARGET_COLUMN])
test_metrics = None if test_df is None else pipe.evaluate(test_df[FEATURE_COLUMNS], test_df[TARGET_COLUMN])
print(json.dumps({'mode': 'zero-shot-icl (in-context conditioning, no gradient update)', 'device': pipe.device, 'source': pipe.source, 'n_estimators': N_ESTIMATORS, 'seed': SEED, 'target_stats': pipe.target_stats, 'validation': validation_metrics, 'test': test_metrics}, indent=2))

## 7. Training-mean baseline → evaluation report

`mean_baseline` always predicts the training mean and is scored on the same validation rows, so the comparison uses identical rows; a model that does not beat it has learned nothing usable. `evaluation_report` is the module's public evaluation stage: it carries the four metrics with the verdict `sample-sanity`, the baseline, the split and the fresh-boundary reload check (filled in by Section 8 and re-written there); without a scored validation split the verdict is `not-measurable`. It is written to `outputs/tabpfn_regressor_evaluation_report.json`. On the synthetic sample these are sanity metrics for the plumbing, not evidence of tabular-regression skill.

In [ ]:
baseline = mean_baseline(train_df[TARGET_COLUMN], val_df[TARGET_COLUMN])
report = evaluation_report(validation_metrics, baseline=baseline, n_validation=len(val_df), target_column=TARGET_COLUMN, sample_kind=sample_kind)
with open('outputs/tabpfn_regressor_evaluation_report.json', 'w', encoding='utf-8') as handle:
    json.dump(report, handle, indent=2, ensure_ascii=False)
print(json.dumps({key: report[key] for key in ('verdict', 'reason', 'adaptation', 'decision_rule', 'n_validation', 'metrics', 'baselines')}, indent=2))
if validation_metrics['mae'] > baseline['mae']:
    print('WARNING: in-context TabPFN does not beat the training-mean baseline on this holdout; inspect the data before drawing any conclusion.')

## 8. Export the artifact bundle and verify it across a fresh boundary

The deployable artifact is the pair `model.tabpfn_fit` (fitted estimator state **including the in-context training rows**) + `model.ckpt` (a byte copy of the verified foundation checkpoint) described by `artifact_manifest.json` (task, target, ordered feature columns, target statistics, per-file SHA-256, base-model identity); the fitted archive alone is not a model. `pipe.save_artifact` writes it and `zip_artifact_bundle` zips it to `outputs/tabpfn_regressor_artifact.zip` for the companion artifact-inference notebook. Then it does what a downstream consumer would do with the bundle and nothing else: copy it to a **fresh location**, `validate_artifact_bundle` (schema, member names, sizes, digests, archive safety — before any model state is deserialised), reconstruct through `TabPFNRegressorPipeline.from_artifact` (the fitted archive's recorded `model_path` is rewritten in a temporary copy to the companion checkpoint; TabPFN's `load_fitted_tabpfn_model` restores the estimator; no refit, no download) and require the recomputed validation MAE to equal the recorded value within `1e-6`. Because the bundle contains training rows, treat it with the same confidentiality controls as the dataset; loading it executes trusted serialised Python/torch state.

In [ ]:
ARTIFACT_DIR = WORK / 'artifact'
artifact_manifest = pipe.save_artifact(ARTIFACT_DIR)
bundle_zip_sha256 = zip_artifact_bundle(ARTIFACT_DIR, 'outputs/tabpfn_regressor_artifact.zip')
print({'artifact': sorted(p.name for p in ARTIFACT_DIR.iterdir()), 'fittedEstimatorSha256': artifact_manifest['fittedEstimatorSha256'][:16], 'foundationCheckpointSha256': artifact_manifest['foundationCheckpointSha256'][:16], 'bundle_zip_sha256': bundle_zip_sha256[:16]})
if artifact_manifest['foundationCheckpointSha256'] != WEIGHTS_SHA256:
    raise RuntimeError('the bundled checkpoint is not the pinned foundation checkpoint')
FRESH_DIR = WORK / 'fresh-reload'
shutil.copytree(ARTIFACT_DIR, FRESH_DIR)
checked = validate_artifact_bundle(FRESH_DIR, expected_checkpoint_sha256=WEIGHTS_SHA256)
fresh = TabPFNRegressorPipeline.from_artifact(FRESH_DIR, device=pipe.device, expected_checkpoint_sha256=WEIGHTS_SHA256)
if fresh.feature_columns != FEATURE_COLUMNS or fresh.target_column != TARGET_COLUMN:
    raise RuntimeError('reloaded artifact disagrees with the validated schema')
reloaded_metrics = fresh.evaluate(val_df[FEATURE_COLUMNS], val_df[TARGET_COLUMN])
reload_check = {'recordedMae': validation_metrics['mae'], 'reloadedMae': reloaded_metrics['mae'], 'tolerance': 1e-6, 'recordedModelPath': checked['recordedModelPath']}
reload_check['maeMatches'] = abs(reload_check['reloadedMae'] - reload_check['recordedMae']) <= reload_check['tolerance']
print(json.dumps(reload_check, indent=2))
if not reload_check['maeMatches']:
    raise RuntimeError('The reloaded artifact does not reproduce the recorded validation metric. Do not ship this artifact.')
report = evaluation_report(validation_metrics, baseline=baseline, n_validation=len(val_df), target_column=TARGET_COLUMN, sample_kind=sample_kind, reload_check=reload_check)
with open('outputs/tabpfn_regressor_evaluation_report.json', 'w', encoding='utf-8') as handle:
    json.dump(report, handle, indent=2, ensure_ascii=False)
print(f'Fresh-boundary verification PASSED on {len(val_df)} validation rows.')

## 9. Predict new rows

Real use means rows the estimator has not seen. The default takes the first eight rows of `test.csv` (or of `val.csv` when no test split exists) with the target column removed; set `USE_BYOD_ROWS = True` to supply your own CSV instead (`NEW_DATA_PATH` for an executor, or the upload dialog) with exactly the artifact's feature columns and no target/`prediction` columns — `validate_new_rows` rejects duplicates, missing or extra columns and infinite values rather than silently dropping anything. `predict` on the **reloaded** estimator returns `prediction`, TabPFN's point estimate in target units; **no prediction interval is shipped**, and a prediction far outside the training target range is an extrapolation the input manifest would have flagged.

In [ ]:
USE_BYOD_ROWS = False  # @param {type:"boolean"}
NEW_DATA_PATH = ''  # @param {type:"string"}

if USE_BYOD_ROWS:
    if NEW_DATA_PATH:
        new_name, new_payload = Path(NEW_DATA_PATH).name, Path(NEW_DATA_PATH).read_bytes()
    else:
        from google.colab import files
        uploaded = files.upload()
        if len(uploaded) != 1:
            raise RuntimeError('Upload exactly one CSV of new rows.')
        new_name, new_payload = next(iter(uploaded.items()))
    new_rows = pd.read_csv(io.BytesIO(new_payload))
    new_rows_origin = f'user-supplied CSV (BYOD): {new_name}'
else:
    source_df, source_label = (test_df, 'test.csv') if test_df is not None else (val_df, 'val.csv')
    new_rows = source_df.drop(columns=[TARGET_COLUMN]).head(8).reset_index(drop=True)
    new_rows_origin = f'first 8 rows of {source_label} (held out from the in-context support), target removed'
new_rows = validate_new_rows(new_rows, FEATURE_COLUMNS, target_column=TARGET_COLUMN)
predictions = fresh.predict(new_rows)
print('scored rows drawn from:', new_rows_origin)
print(predictions.to_string(index=False))

## 10. Export machine-readable results and provenance

`outputs/tabpfn_regressor_predictions.csv` holds one row per scored input (`row_id`, `prediction`); `outputs/tabpfn_regressor_result.json` records the validation and test metrics, the training-mean baseline, the evaluation report, the fresh-boundary reload check, the input manifest, the dataset identity (ZIP digest, split origin, row counts), the artifact manifest (with its digests), the inference settings, the notebook's source (repository, revision, module digest, generator), the pinned model identity, revision and licence, and the runtime. `outputs/tabpfn_regressor_artifact.zip` is the bundle for the companion notebook. No credentials are recorded.

In [ ]:
predictions.to_csv('outputs/tabpfn_regressor_predictions.csv', index=False)
payload = {
    'metrics': {'validation': validation_metrics, 'test': test_metrics},
    'training_mean_baseline': baseline,
    'evaluation_report': report,
    'fresh_boundary_reload': reload_check,
    'input_manifest': input_manifest,
    'dataset': {**dataset_origin, 'sample_kind': sample_kind, 'zip_sha256': dataset_sha256, 'target_column': TARGET_COLUMN, 'feature_columns': FEATURE_COLUMNS, 'splits': split_origin, 'rows': {'train': len(train_df), 'val': len(val_df), 'test': None if test_df is None else len(test_df)}},
    'artifact': artifact_manifest,
    'inference': {'mode': 'zero-shot-icl', 'adaptation': 'in-context conditioning only; the private-worker fine-tune path is not carried', 'n_estimators': N_ESTIMATORS, 'seed': SEED, 'decision_rule': DECISION_RULE, 'new_rows_origin': new_rows_origin, 'scored_rows': int(len(predictions))},
    'notebook_source': NOTEBOOK_SOURCE,
    'repository_revision': NOTEBOOK_SOURCE['repository_revision'],
    'model_id': MODEL_ID,
    'model_revision': MODEL_REVISION,
    'model_license': MODEL_LICENSE,
    'model_file': WEIGHTS_FILE,
    'runtime': {'python': platform.python_version(), 'torch': torch.__version__, 'tabpfn': importlib.metadata.version('tabpfn'), 'pandas': pd.__version__, 'scikit_learn': importlib.metadata.version('scikit-learn'), 'device': pipe.device},
}
with open('outputs/tabpfn_regressor_result.json', 'w', encoding='utf-8') as handle:
    json.dump(payload, handle, indent=2, ensure_ascii=False, default=str)
print(sorted(os.listdir('outputs')))

## Interpretation and limits

The prediction is TabPFN's point estimate — the mean of its predictive distribution — with no shipped prediction interval; treat it as a best guess, not a range. The evaluation report's `sample-sanity` verdict names what it is: one holdout of one table with no dispersion estimate — on the synthetic sample a plumbing check, and even BYOD metrics must not be generalised to a domain, a population or a target range. Random splitting assumes independent rows; temporal, grouped or patient-level data need leakage-safe splits you supply. MAPE is undefined around zero and is reported over non-zero targets only. In-context learning is not fine-tuning: the estimator's quality depends entirely on the training rows it conditions on, and the artifact carries those rows. Digest equality proves the checkpoint bytes are the ones pinned at the immutable revision; it does not by itself prove who published them.

Successful execution proves that the recorded repository revision's pipeline module, carried in this notebook, can acquire and digest-verify the pinned TabPFN-3 checkpoint, validate the demonstrated table into an input manifest, fit in context and beat a trivial baseline on the synthetic sample, export the artifact bundle, reload it across a fresh boundary and reproduce the recorded metric, predict new rows, and emit the shown machine-readable outputs in the tested runtime — without the repository being reachable. It does **not** establish benchmark superiority, generalisation, robustness, calibrated uncertainty, safety for high-consequence decisions, production fitness, or anything about the fine-tuned models the private DIMER worker produces.

**Next experiments:** raise `N_ESTIMATORS` to 8 and compare RMSE; enable `USE_BYOD` with a small table of your own (a few hundred rows, a numeric target) and read R² against the training-mean baseline; drop `val.csv` from the ZIP and watch the seeded holdout take over; hand `outputs/tabpfn_regressor_artifact.zip` to the companion artifact-inference notebook in a fresh session.

## References

- Repository README: https://github.com/kurtvalcorza/tabpfn-regressor-pipeline/blob/main/README.md
- Repository model card: https://github.com/kurtvalcorza/tabpfn-regressor-pipeline/blob/main/MODEL_CARD.md
- Weight provenance: https://github.com/kurtvalcorza/tabpfn-regressor-pipeline/blob/main/docs/WEIGHTS.md
- Upstream model: https://huggingface.co/Prior-Labs/tabpfn_3
- Upstream code: https://github.com/PriorLabs/TabPFN
- TabPFN-3 technical report: https://arxiv.org/abs/2605.13986